# 第4课：系统评估与策略对比## 本节目标没有评估就没有改进方向。本节展示如何量化衡量 RAG 系统的各个环节。你将学会：1. **检索质量评估**：延迟、BM25-向量重叠度、分数方差2. **分块策略对比**：不同策略对检索效果的系统性影响3. **嵌入模型对比**：不同模型在相同查询下的表现差异4. **生成质量评估**：回答长度、来源引用数5. **完整评估报告**：一键运行所有评估5. **���Զ������Ա���**��ʹ�� pytest �Զ���������֤����ģ�飬����ʵ��ͼ����
> 评估是 RAG 系统优化的指南针——能量化的就不要靠感觉。

## 环境准备

In [ ]:
import syssys.path.insert(0, '..')from config import configfrom src.pipeline import RAGPipelinefrom src.evaluator import Evaluatorprint("模块加载成功!")

## 1. 初始化流水线和评估器

In [ ]:
# 构建流水线pipeline = RAGPipeline()pipeline.build_index()# 创建评估器evaluator = Evaluator(pipeline)print(f"准备评估，共 {len(config.eval_queries)} 条测试查询")

## 2. 检索性能评估评估指标说明：- **延迟（Latency）**：从提问到返回检索结果的时间- **BM25-向量重叠度（Jaccard）**：两种检索方式的一致性——越高说明两者越一致- **分数方差**：检索结果分数的离散程度——方差小说明结果质量均匀

In [ ]:
retrieval_eval = evaluator.evaluate_retrieval()print(f"检索性能报告:")print(f"  测试查询数: {retrieval_eval['num_queries']}")print(f"  平均延迟: {retrieval_eval['avg_latency_ms']}ms")print(f"  BM25-向量平均重叠度: {retrieval_eval['avg_bm25_vector_overlap']:.4f}")print(f"  平均分数方差: {retrieval_eval['avg_score_variance']:.6f}")print(f"\n  逐查询详情:")for r in retrieval_eval['per_query']:    print(f"    [{r['latency_ms']:4d}ms] {r['query'][:50]}...")

## 3. 分块策略系统性对比这是本教程最有价值的实验之一。我们让三种分块策略在相同的10条查询上运行，对比它们产生的块数、平均块长、检索延迟和重叠度。

In [ ]:
print("对比分块策略（可能需要1-2分钟）...")chunking_eval = evaluator.evaluate_chunking_strategies()print(f"\n{'策略':<20} {'块数':<8} {'平均长度':<10} {'延迟(ms)':<12} {'重叠度':<10}")print("-" * 60)for strategy, stats in chunking_eval.items():    print(f"{strategy:<20} {stats['num_chunks']:<8} "          f"{stats['avg_chunk_length']:<10.0f} "          f"{stats['avg_latency_ms']:<12} "          f"{stats['avg_bm25_vector_overlap']:<10.4f}")

### 如何解读结果？- **块数太多**（fixed_token）：检索速度变慢，且可能返回过于碎片化的内容- **块数太少**：每个块包含太多信息，LLM 难以聚焦- **重叠度走低**：说明该策略导致 BM25 和向量检索的共识减少，可能影响混合检索效果通常 `recursive_char` 是"甜区"——它在语义完整性和块大小之间取得平衡。

## 4. 嵌入模型对比同一个查询，不同嵌入模型返回的结果可能完全不同。

In [ ]:
embedding_eval = evaluator.evaluate_embedding_models()if embedding_eval:    print(f"\n{'模型':<35} {'维度':<8} {'平均相似度':<12} {'覆盖率':<10}")    print("-" * 65)    for model, stats in embedding_eval.items():        coverage = stats.get('coverage', 0)        print(f"{model:<35} {stats['dim']:<8} "              f"{stats['avg_similarity']:<12.4f} {coverage:<10.4f}")    best_model = max(embedding_eval, key=lambda m: embedding_eval[m]['avg_similarity'])    print(f"\n平均相似度最高: {best_model}")else:    print("嵌入模型对比需要先安装 sentence-transformers")

## 5. 生成质量评估

In [ ]:
gen_eval = evaluator.evaluate_generation_quality()print(f"生成质量报告:")print(f"  测试查询数: {gen_eval['num_queries']}")print(f"  平均回答长度: {gen_eval['avg_answer_length']:.0f} 字符")print(f"  平均引用来源: {gen_eval['avg_sources_used']:.1f} 个")for r in gen_eval['per_query']:    print(f"\n  Q: {r['query']}")    print(f"  回答长度: {r['answer_length']} 字符, "          f"来源数: {r['num_sources']}, 耗时: {r['retrieval_time_ms']}ms")

## 6. 一键完整评估调用 `run_full_evaluation()` 运行所有评估并打印报告。

In [ ]:
# 完整评估（需要 API Key 以获得 LLM 生成评估）results = evaluator.run_full_evaluation()# 打印格式化报告evaluator.print_report()

## 7. 优化建议总结基于评估结果，RAG 系统的优化通常从以下维度入手：| 维度 | 方法 | 预期效果 ||------|------|---------|| 分块 | 调优 chunk_size 和 overlap | 影响检索精度和生成连贯性 || 嵌入 | 换更强的模型（如 BGE-large） | 提升语义匹配准确度 || 检索 | 调优 BM25/Vector 权重比 | 平衡关键词和语义匹配 || 融合 | 从 RRF 切换到加权融合 | 更灵活的控制 || Prompt | 优化角色设定和格式约束 | 提升生成质量和可用性 || Re-Rank | 使用 BGE-Reranker 重排序 | 显著提升 Top-3 精度 |> 提示：调优 RAG 是系统性工作——改一个参数可能影响多个指标。建议每次只改一个变量，对比前后效果。

## 8. 自动化测试

RAG-Lab 项目包含完整的 **pytest** 自动化测试套件（54 个用例），覆盖四个核心模块。

| 测试文件 | 用例数 | 覆盖内容 |
|---------|-------|---------|
| test_chunker.py | 17 | 三种分块策略的正确性、边界条件、元数据完整性 |
| test_embedder.py | 6 | 向量维度、L2 归一化、一致性、区分度 |
| test_retriever.py | 22 | 中英文检测与分词、RRF/加权融合数学、BM25/向量/混合检索集成 |
| test_vector_store.py | 9 | ChromaDB 增删查、清除重建、FAISS（可选） |

### 运行测试

在项目根目录的终端中执行：

    pytest tests/ -v                        # 全部测试
    pytest tests/ -v -k "not Integration"   # 仅快速单元测试
    pytest tests/ --cov=src --cov-report=term-missing  # 覆盖率报告

测试采用离线模式加载模型（HF_HUB_OFFLINE=1），从本地缓存即时加载，不依赖网络。


In [ ]:
# 在 notebook 中运行测试（需要在项目根目录下执行）
import sys, os, subprocess
sys.path.insert(0, "..")

# 以下命令需要在项目根目录的终端中运行，这里仅作演示
os.chdir("..")
result = subprocess.run(
    ["python", "-m", "pytest", "tests/", "-v", "--tb=short"],
    capture_output=True, text=True, cwd=".."
)
# 打印测试结果摘要
lines = result.stdout.split("
")
for line in lines[-15:]:
    if line.strip():
        print(line)


## 9. 实验图表生成

项目提供了图表生成脚本 scripts/generate_figures.py，基于评估数据自动生成实验报告所需的 PDF 图表。

### 生成的图表

| 图表文件 | 内容 |
|---------|------|
| chunking_comparison.pdf | 三种分块策略的分块数、平均长度、延迟对比 |
| retrieval_performance.pdf | 逐查询的 BM25-向量重叠度与检索延迟 |
| embedding_comparison.pdf | 三种嵌入模型的相似度、覆盖率、维度对比 |
| hybrid_quality.pdf | 混合检索 Top-1 得分与 Top-5 平均得分 |
| retrieval_methods.pdf | 三种检索方式两两之间的 Jaccard 重叠度 |

### 生成方式

在项目根目录的终端中执行：

    python scripts/generate_figures.py

脚本会运行完整评估流程，收集实际数据，使用 matplotlib 生成矢量 PDF 图表。图表可直接用于 LaTeX 实验报告。


In [ ]:
# 查看已生成的图表文件
import os
fig_dir = os.path.join("..", "figures")
if os.path.isdir(fig_dir):
    files = sorted(os.listdir(fig_dir))
    print(f"图表目录: {fig_dir}")
    for f in files:
        size_kb = os.path.getsize(os.path.join(fig_dir, f)) / 1024
        print(f"  {f:35s} {size_kb:6.1f} KB")
else:
    print("图表目录不存在，请先运行: python scripts/generate_figures.py")


## 本节小结- 评估是 RAG 系统优化的前提——量化后才知道改进了多少- 分块策略对检索效果有显著影响，`recursive_char` 通常是最佳起点- 嵌入模型的选择需要平衡精度和速度- 一键评估让你快速获得全貌，逐项评估让你深入每个细节---## 恭喜你完成了 RAG-Lab 全部教程！你现在已经理解了 RAG 系统的完整架构和每个环节的作用。建议接下来：1. 用 `python main.py ui` 启动网页界面，交互式体验2. 修改 `config.py` 中的参数，观察影响3. 替换为自己的文档集（修改 `document_loader.py` 中的数据源）4. 尝试添加 Re-Ranker 或 Query 改写等进阶功能